In [0]:
import re
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

catalog = "severn_trent"
schema = "bronze"
raw_data_path = "/Volumes/severn_trent/bronze/raw_data"

# Scan the raw data volume to discover snapshot directories matching DD_MM_YYYY pattern
snapshot_folders = [
    folder.path.rstrip("/")
    for folder in dbutils.fs.ls(raw_data_path)
    if folder.isDir()
    and re.fullmatch(r"\d{2}_\d{2}_\d{4}", folder.name.rstrip("/"))
]

if not snapshot_folders:
    raise ValueError(f"No snapshot folders found in {raw_data_path}")

latest_snapshot_path = max(
    snapshot_folders,
    key=lambda path: datetime.strptime(path.split("/")[-1], "%d_%m_%Y")
)

latest_snapshot = latest_snapshot_path.split("/")[-1]

# List all CSV source files present within the latest snapshot directory
source_files = [
    file.path
    for file in dbutils.fs.ls(latest_snapshot_path)
    if file.name.lower().endswith(".csv")
]

for source_file_path in source_files:
    file_name = source_file_path.split("/")[-1]
    source_name = file_name.replace(".csv", "")

    target_table_name = re.sub(r"_\d{2}_\d{2}_\d{4}$", "", source_name)
    target_table = f"{catalog}.{schema}.{target_table_name}"

    # Skip a file that has already been ingested.
    # checks for the file path in the table.
    if spark.catalog.tableExists(target_table):
        already_ingested = (
            spark.table(target_table)
            .filter(col("source_file_path") == source_file_path)
            .limit(1)
            .count() > 0
        )

        if already_ingested:
            print(f"Skipping already ingested file: {source_file_path}")
            continue

    print(f"Processing file: {file_name}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_file_path)
        .withColumn("source_file_path", lit(source_file_path))
        .withColumn("last_update_ts", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Completed: {target_table}")

print(f"Latest snapshot {latest_snapshot} processed.")